# bench_ssl — Semi-Supervised Learning Benchmark
> Run on Kaggle with 2×T4 GPUs.

Benchmarks SSL algorithms across two modalities:
- **Vision**: CIFAR-10 with MeanTeacher, PseudoLabel, PiModel vs supervised baseline.
- **Tabular**: Forest Covertype with MeanTeacher and PseudoLabel vs supervised baseline.

`SSLVisionWrapper` and `SSLTabularWrapper` live in `ml_pipeline/pipelines_torch/ssl_wrappers.py`.


## Setup


In [ ]:
import os, subprocess, sys
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "ml_pipeline" else cwd
if (repo_root / "ml_pipeline").is_dir() and (repo_root / "requirements.txt").is_file():
    os.chdir(repo_root)
    print(f"Using local checkout: {repo_root}")
else:
    subprocess.run(["rm", "-rf", "bench_research_ml_project"], check=True)
    subprocess.run(["git", "clone", "https://github.com/oremaz/bench_research_ml_project"], check=True)
    os.chdir("bench_research_ml_project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print(f"Working directory: {Path.cwd()}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Import wrappers from the module — no inline class definitions needed
from ml_pipeline.pipelines_torch.ssl_wrappers import SSLVisionWrapper, SSLTabularWrapper
from ml_pipeline.pipelines_torch.benchmark import BenchmarkRunner

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Vision — CIFAR-10
We compare four models on CIFAR-10:
- **Supervised baseline** — standard cross-entropy on 2 000 labelled samples only.
- **MeanTeacher** — consistency regularisation via an EMA teacher.
- **PseudoLabel** — high-confidence unlabelled predictions added to the supervised loss.
- **PiModel** — two stochastic forward passes encouraged to agree.


### 1.1 Dataset Preparation


In [ ]:
print("Loading CIFAR-10 ...")
full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
X_vis = np.transpose(full_train.data, (0, 3, 1, 2)).astype(np.float32) / 255.0
X_vis = (X_vis - 0.5) / 0.5  # Normalise to [-1, 1]
y_vis = np.array(full_train.targets)

# Subsample to 15 000 for faster iteration
X_vis, _, y_vis, _ = train_test_split(
    X_vis, y_vis, train_size=15_000, stratify=y_vis, random_state=42
)
print(f"CIFAR-10 subset: {X_vis.shape}, {y_vis.shape}")


### 1.2 Benchmark


In [ ]:
vision_models = [
    {"name": "Vision_Supervised",  "class": SSLVisionWrapper, "params": {"ssl_algo": "supervised"}},
    {"name": "Vision_MeanTeacher", "class": SSLVisionWrapper, "params": {"ssl_algo": "mean_teacher"}},
    {"name": "Vision_PseudoLabel", "class": SSLVisionWrapper, "params": {"ssl_algo": "pseudo_label"}},
    {"name": "Vision_PiModel",     "class": SSLVisionWrapper, "params": {"ssl_algo": "pi_model"}},
]

runner_vis = BenchmarkRunner(
    model_configs=vision_models,
    augmentations=[None],
    task_type="classification",
    device=str(device),
    epochs=5,
    batch_size=32,
    use_kfold=False,
    path_start="ssl_vision",
)

print("\n--- Vision SSL Benchmark ---")
runner_vis.run(X_vis, y_vis)


### 1.3 Analysis


In [ ]:
import os, glob, json, pandas as pd

vis_results = []
for p in glob.glob("results/ssl_vision/**/metrics.json", recursive=True):
    with open(p) as f:
        d = json.load(f)
    d["model"] = os.path.basename(os.path.dirname(p))
    vis_results.append(d)

vis_df = pd.DataFrame(vis_results)
print(vis_df)

if not vis_df.empty and "accuracy" in vis_df.columns:
    vis_df.set_index("model")["accuracy"].sort_values().plot(
        kind="barh", title="CIFAR-10 Test Accuracy — Vision SSL", xlim=(0, 1)
    )
    plt.tight_layout(); plt.show()


## 2. Tabular — Forest Covertype
Only **500 samples** are treated as labelled; the remaining 79 500 are unlabelled (~0.6% label ratio).
This is where SSL algorithms are expected to shine most.


### 2.1 Dataset Preparation


In [ ]:
from sklearn.datasets import fetch_covtype

print("Fetching Forest Covertype dataset ...")
covtype = fetch_covtype()
X_tab_raw = covtype.data
y_tab_raw = covtype.target - 1  # labels 1-7 → 0-6

# Subsample to 20 000 for speed
X_tab_raw, _, y_tab_raw, _ = train_test_split(
    X_tab_raw, y_tab_raw, train_size=20_000, stratify=y_tab_raw, random_state=42
)
X_tab_raw = StandardScaler().fit_transform(X_tab_raw)
print(f"Covertype subset: {X_tab_raw.shape} | classes: {len(set(y_tab_raw))}")


### 2.2 Benchmark


In [ ]:
tabular_models = [
    {"name": "Tabular_Supervised",  "class": SSLTabularWrapper,
     "params": {"ssl_algo": "supervised", "input_dim": X_tab_raw.shape[1], "num_classes": 7}},
    {"name": "Tabular_MeanTeacher", "class": SSLTabularWrapper,
     "params": {"ssl_algo": "mean_teacher", "input_dim": X_tab_raw.shape[1], "num_classes": 7}},
    {"name": "Tabular_PseudoLabel", "class": SSLTabularWrapper,
     "params": {"ssl_algo": "pseudo_label", "input_dim": X_tab_raw.shape[1], "num_classes": 7}},
]

runner_tab = BenchmarkRunner(
    model_configs=tabular_models,
    augmentations=[None],
    task_type="classification",
    device=str(device),
    epochs=10,
    batch_size=32,
    use_kfold=False,
    path_start="ssl_tabular",
)

print("\n--- Tabular SSL Benchmark ---")
runner_tab.run(X_tab_raw, y_tab_raw)


### 2.3 Analysis


In [ ]:
tab_results = []
for p in glob.glob("results/ssl_tabular/**/metrics.json", recursive=True):
    with open(p) as f:
        d = json.load(f)
    d["model"] = os.path.basename(os.path.dirname(p))
    tab_results.append(d)

tab_df = pd.DataFrame(tab_results)
print(tab_df)

if not tab_df.empty and "accuracy" in tab_df.columns:
    tab_df.set_index("model")["accuracy"].sort_values().plot(
        kind="barh", title="Covertype Test Accuracy — Tabular SSL", xlim=(0, 1)
    )
    plt.tight_layout(); plt.show()


## 3. Summary

| Setting | Supervised | MeanTeacher | PseudoLabel | PiModel |
| --- | --- | --- | --- | --- |
| CIFAR-10 (vision) | *see above* | *see above* | *see above* | *see above* |
| Covertype (tabular) | *see above* | *see above* | *see above* | N/A |

**Takeaways**:
- SSL benefit is most visible when the labelled set is extremely small (500 / 80 000 ≈ 0.6%).
- MeanTeacher typically outperforms PseudoLabel at low label ratios due to smoother consistency targets.
- The wrappers in `ssl_wrappers.py` can be reused directly: swap `input_dim`, `num_classes`, or `ssl_algo`.
